#### shp 내부 값으로 기름 구분 최적 임계값 찾기 (valley + Otsu)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import rasterio
from rasterio.features import geometry_mask
import fiona
import matplotlib.pyplot as plt

# ============================================
# 1. 입력 경로 설정
# ============================================
RASTER_PATH = r"<DATA_ROOT>\07_disaster\oil_spill\snap\S1A_IW_GRDH_1SDV_20250318T233746_20250318T233820_058365_073776_BBC3_Orb_tnr_Bdr_Cal_Spk_TC_dB.tif"
SHAPE_PATH  = r"<DATA_ROOT>\07_disaster\oil_spill\shp\Oil.shp"
BAND_INDEX  = 2   # VV 밴드 번호 (1-based index)


def extract_clean_values(raster_path, shp_path, band_index):
    """래스터 + shp에서 NaN/nodata/0 제거한 shp 내부 값만 추출"""
    with rasterio.open(raster_path) as src:
        band = src.read(band_index).astype("float32")
        raster_crs = src.crs
        transform = src.transform
        height, width = src.height, src.width
        nodata = src.nodata

    with fiona.open(shp_path, "r") as shp:
        shp_crs = shp.crs
        geoms = [feat["geometry"] for feat in shp]

    if str(shp_crs) != str(raster_crs):
        raise ValueError(f"CRS가 다릅니다. raster={raster_crs}, shp={shp_crs}")

    sea_mask = geometry_mask(
        geoms,
        transform=transform,
        invert=True,
        out_shape=(height, width)
    )

    inside_values = band[sea_mask]

    valid = np.isfinite(inside_values)

    if nodata is not None:
        valid &= (inside_values != nodata)

    valid &= (inside_values != 0)

    values_clean = inside_values[valid]
    return values_clean


def threshold_valley(vv, bins=200):
    """두 peak 사이 valley(히스토그램 최소값)에 기반한 threshold"""
    hist, bin_edges = np.histogram(vv, bins=bins)

    # 가장 높은 두 개 peak 위치
    peaks = np.argpartition(hist, -2)[-2:]
    p1, p2 = np.sort(peaks)

    # 두 peak 사이에서 최소 빈도 위치 → valley
    valley_idx = np.argmin(hist[p1:p2]) + p1
    thr_valley = bin_edges[valley_idx]

    return thr_valley, hist, bin_edges


def threshold_otsu_manual(vv, bins=200):
    """skimage 없이 Otsu threshold 직접 구현"""
    hist, bin_edges = np.histogram(vv, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    weight1 = np.cumsum(hist)
    weight2 = np.sum(hist) - weight1

    # 0 division 방지
    valid = (weight1 > 0) & (weight2 > 0)

    mean1 = np.zeros_like(bin_centers)
    mean2 = np.zeros_like(bin_centers)

    mean1[valid] = np.cumsum(hist * bin_centers)[valid] / weight1[valid]
    total_mean = np.sum(hist * bin_centers)
    mean2[valid] = (total_mean - np.cumsum(hist * bin_centers)[valid]) / weight2[valid]

    # 클래스 간 분산
    variance = weight1[:-1] * weight2[:-1] * (mean1[:-1] - mean2[:-1])**2

    thr_otsu = bin_centers[:-1][np.argmax(variance)]
    return thr_otsu


def main():
    # -----------------------------
    # 1) shp 내부 값 추출 (NaN/nodata/0 제거)
    # -----------------------------
    vv = extract_clean_values(RASTER_PATH, SHAPE_PATH, BAND_INDEX)
    print("유효 픽셀 수 (NaN/nodata/0 제거 후):", vv.size)

    if vv.size == 0:
        print("유효한 픽셀이 없습니다. 입력 데이터/마스크를 확인하세요.")
        return

    print("\n=== shp 내부 통계 (NaN/nodata/0 제거 후) ===")
    print("min :", float(vv.min()))
    print("max :", float(vv.max()))
    print("mean:", float(vv.mean()))
    print("std :", float(vv.std()))

    # -----------------------------
    # 2) Valley 기반 threshold
    # -----------------------------
    thr_valley, hist, bin_edges = threshold_valley(vv, bins=200)
    print("\nValley-based threshold:", thr_valley)

    # -----------------------------
    # 3) Otsu 기반 threshold
    # -----------------------------
    thr_otsu = threshold_otsu_manual(vv, bins=200)
    print("Otsu threshold          :", thr_otsu)

    # -----------------------------
    # 4) 히스토그램 + 임계값 시각화
    # -----------------------------
    plt.figure(figsize=(8, 5))
    plt.hist(vv, bins=200, alpha=0.7)

    plt.axvline(thr_valley, color='r', linestyle='--', linewidth=2, label=f"Valley: {thr_valley:.2f}")
    plt.axvline(thr_otsu,  color='g', linestyle='-.', linewidth=2, label=f"Otsu: {thr_otsu:.2f}")

    plt.title("VV Histogram within SHP + Oil/Non-Oil Thresholds")
    plt.xlabel("Backscatter (dB)")
    plt.ylabel("Pixel count")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import rasterio
from pathlib import Path
from scipy.ndimage import gaussian_filter
import fiona
from rasterio.features import geometry_mask


# ============================================
# ★ 1. 사용자가 수정할 부분 ★
# ============================================
INPUT_TIF     = r"<DATA_ROOT>\07_disaster\oil_spill\oil_detection.tif"   # 입력 래스터
SHP_PATH      = r"<DATA_ROOT>\07_disaster\oil_spill\shp\Oil.shp"        # 관심 영역 shp

BAND_INDEX    = 2              # 래스터에서 사용할 밴드 번호 (1-based index)
THRESHOLD     = -27.7          # 임계값: 이 값 이하 → 1

OUTPUT_MASK   = r"<DATA_ROOT>\07_disaster\oil_spill\oil_mask_binary.tif"
OUTPUT_GAUSS  = r"<DATA_ROOT>\07_disaster\oil_spill\oil_mask_gauss_bin.tif"

GAUSS_SIGMA   = 2.0            # Gaussian blur sigma
GAUSS_BIN_THR = 0.5            # 블러 결과를 0/1로 다시 threshold할 기준 (보통 0.5 전후)
# ============================================


def main():
    in_path = Path(INPUT_TIF)
    if not in_path.exists():
        raise FileNotFoundError(f"입력 래스터 없음: {in_path}")

    shp_path = Path(SHP_PATH)
    if not shp_path.exists():
        raise FileNotFoundError(f"입력 Shapefile 없음: {shp_path}")

    # ----------------------------------------
    # 2. 래스터 읽기 (특정 밴드만)
    # ----------------------------------------
    with rasterio.open(in_path) as src:
        print("Input raster:", in_path)
        print("Raster shape:", src.height, src.width)
        print("Total bands :", src.count)
        print("Using band  :", BAND_INDEX)

        data = src.read(BAND_INDEX).astype("float32")  # 특정 밴드만 읽기
        profile = src.profile.copy()
        nodata = src.nodata
        raster_crs = src.crs
        transform = src.transform
        height, width = src.height, src.width

    # ----------------------------------------
    # 3. Shapefile 읽고, 래스터와 CRS 확인
    # ----------------------------------------
    with fiona.open(shp_path, "r") as shp:
        shp_crs = shp.crs
        geoms = [feat["geometry"] for feat in shp]

    print("Raster CRS    :", raster_crs)
    print("Shapefile CRS :", shp_crs)

    if str(shp_crs) != str(raster_crs):
        raise ValueError(f"CRS가 다릅니다. raster={raster_crs}, shp={shp_crs}")

    # ----------------------------------------
    # 4. shp 기반 영역 마스크 생성
    #    True: shp 내부, False: 외부
    # ----------------------------------------
    shp_mask = geometry_mask(
        geoms,
        transform=transform,
        invert=True,            # polygon 내부를 True로
        out_shape=(height, width)
    )

    print("shp_mask shape:", shp_mask.shape)

    # ----------------------------------------
    # 5. 유효 픽셀 마스크 (NaN / nodata 제거 + shp 내부만)
    # ----------------------------------------
    valid = np.isfinite(data)
    if nodata is not None:
        valid &= (data != nodata)

    # shp 내부만 남기기
    valid &= shp_mask

    # ----------------------------------------
    # 6. Threshold 기반 binary mask (<= threshold → 1)
    #    (shp 내부 + 유효 픽셀에서만 적용)
    # ----------------------------------------
    mask_bool = valid & (data <= THRESHOLD)

    # float32로 만들고 나중에 uint8로 변환
    mask = np.zeros_like(data, dtype="float32")
    mask[mask_bool] = 1.0

    print(f"Threshold applied: <= {THRESHOLD}")
    print("1로 표시된 픽셀 수 (shp 내부):", int(mask_bool.sum()))

    # ----------------------------------------
    # 7. Gaussian Blur (전체 래스터 크기에서 블러)
    # ----------------------------------------
    mask_gauss = gaussian_filter(mask, sigma=GAUSS_SIGMA)

    # ----------------------------------------
    # 8. 블러 결과를 다시 0/1 binary로 threshold
    # ----------------------------------------
    mask_gauss_bin = (mask_gauss >= GAUSS_BIN_THR).astype("uint8")

    # 원래 threshold 마스크도 uint8로
    mask_bin = mask.astype("uint8")

    # ----------------------------------------
    # 9. 출력 profile 설정 (둘 다 0/1 마스크니까 uint8)
    # ----------------------------------------
    out_profile = profile.copy()
    out_profile.update(
        dtype="uint8",
        count=1,         # 항상 1밴드 출력
        nodata=0,        # 0을 nodata/배경으로 써도 무방
        compress="lzw"
    )

    # ----------------------------------------
    # 10. 이진 마스크 저장 (원본 threshold 바로 적용 결과)
    # ----------------------------------------
    with rasterio.open(OUTPUT_MASK, "w", **out_profile) as dst:
        dst.write(mask_bin, 1)

    # ----------------------------------------
    # 11. Gaussian Blur 후 다시 이진화된 마스크 저장
    # ----------------------------------------
    with rasterio.open(OUTPUT_GAUSS, "w", **out_profile) as dst:
        dst.write(mask_gauss_bin, 1)

    print("\n=== 저장 완료 ===")
    print("Binary mask (raw threshold) saved   :", OUTPUT_MASK)
    print("Binary mask (gaussian + threshold)  :", OUTPUT_GAUSS)
    print(f"Gaussian blur sigma = {GAUSS_SIGMA}, binarize threshold = {GAUSS_BIN_THR}")


if __name__ == "__main__":
    main()
